### Data preparation: Train/test splitting (time-aware)

In [1]:
# imports
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

/home/nahshon/envs/general101/lib/python3.12/site-packages/sklearn/utils/__init__.py:16: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.4)
  from scipy.sparse import issparse


In [2]:
# load, sort by time, and define splits
DATA_PATH = '../all_data_files/cleaned_dataset_per_device.csv'
SAVE_DIR = 'Data+Files+Plots+etc'
TEST_SIZE = 0.2  # hold out the most recent 20%

df = pd.read_csv(DATA_PATH, parse_dates=['time'])
df = df.sort_values('time').reset_index(drop=True)

feature_names = ['distance', 'frequency', 'c_walls', 'w_walls', 'co2', 'humidity', 'pm25', 'pressure', 'temperature', 'snr']
target_col = 'exp_pl'

split_idx = int(len(df) * (1 - TEST_SIZE))
train_df = df.iloc[:split_idx].copy()
test_df  = df.iloc[split_idx:].copy()

X_train = train_df[feature_names].to_numpy()
y_train = train_df[target_col].to_numpy()
time_train = train_df['time'].to_numpy()

X_test = test_df[feature_names].to_numpy()
y_test = test_df[target_col].to_numpy()
time_test = test_df['time'].to_numpy()

print(f"Training samples: {len(train_df)}, Test samples: {len(test_df)}")
print(f"Train window: {train_df.time.min()} -> {train_df.time.max()}")
print(f"Test  window: {test_df.time.min()} -> {test_df.time.max()}")

Training samples: 2109044, Test samples: 527262
Train window: 2024-10-01 00:01:07.420593+00:00 -> 2025-07-22 04:48:41.803422+00:00
Test  window: 2025-07-22 04:48:52.796284+00:00 -> 2025-09-30 23:59:55.971870+00:00


In [3]:
# save the time-ordered splits (carry device_id through)

# metadata to carry through outputs
meta_cols = [c for c in ['device_id'] if c in df.columns]

os.makedirs(SAVE_DIR, exist_ok=True)

# base columns to carry forward (meta + features + PL + time)
base_cols = meta_cols + feature_names

train_df_out = train_df[base_cols].copy()
train_df_out['PL'] = y_train
train_df_out['time'] = time_train

test_df_out = test_df[base_cols].copy()
test_df_out['PL'] = y_test
test_df_out['time'] = time_test

train_df_out.to_csv(f"{SAVE_DIR}/train.csv", index=False)
test_df_out.to_csv(f"{SAVE_DIR}/test.csv", index=False)
print(f"Saved train.csv and test.csv to {SAVE_DIR}")

Saved train.csv and test.csv to Data+Files+Plots+etc


In [4]:
# time-series folds for CV (training set)
N_SPLITS = 5
GAP = 0  #  gap between train/val in time series split

tscv = TimeSeriesSplit(n_splits=N_SPLITS, gap=GAP)
fold_assignments = np.zeros(len(train_df), dtype=int)

for fold_num, (_, val_idx) in enumerate(tscv.split(train_df)):
    fold_assignments[val_idx] = fold_num
    val_start, val_end = time_train[val_idx].min(), time_train[val_idx].max()
    print(f"Fold {fold_num+1}: val window {val_start} -> {val_end}")

np.save(f"{SAVE_DIR}/train_folds.npy", fold_assignments)
print(f"Saved time-ordered 5-fold assignments to {SAVE_DIR}/train_folds.npy")

Fold 1: val window 2024-11-18 02:02:43.981560+00:00 -> 2025-01-03 07:42:47.730162+00:00
Fold 2: val window 2025-01-03 07:43:03.618273+00:00 -> 2025-03-02 07:59:49.357312+00:00
Fold 3: val window 2025-03-02 07:59:57.622932+00:00 -> 2025-04-18 14:16:08.491632+00:00
Fold 4: val window 2025-04-18 14:16:10.814991+00:00 -> 2025-06-04 16:12:13.715360+00:00
Fold 5: val window 2025-06-04 16:12:15.965463+00:00 -> 2025-07-22 04:48:41.803422+00:00
Saved time-ordered 5-fold assignments to Data+Files+Plots+etc/train_folds.npy
